In [1]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from os.path import exists

sys.path.append('../..')

In [3]:
from stable_baselines3 import PPO, DQN

In [4]:
from vimms.Common import POSITIVE, load_obj, save_obj
from vimms.Evaluation import evaluate_real
from vimms.Chemicals import ChemicalMixtureFromMZML
from vimms.Roi import RoiBuilderParams

from vimms.MassSpec import IndependentMassSpectrometer
from vimms.Controller import TopNController
from vimms.Environment import Environment

from vimms_gym.env import DDAEnv
from vimms_gym.evaluation import run_method
from vimms_gym.common import METHOD_PPO, METHOD_TOPN, METHOD_RANDOM

# 1. Parameters

In [5]:
mz_range = (70, 1000)
rt_range = (0, 1440)

In [6]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]

In [7]:
isolation_window = 0.7
N = 10
rt_tol = 120
small_rt_tol = 15
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE

In [8]:
params = {
    'chemical_creator': None,
    'noise': {
        'enable_spike_noise': False,
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
    }
}

In [9]:
max_peaks = 200
in_dir = '../QCB_chems/results'

In [10]:
deterministic = True

# 2. Evaluation on QCB data

In [11]:
env_name = 'DDAEnv'

In [12]:
eval_dir = 'evaluation_QCB'
methods = [
    METHOD_PPO,
    METHOD_TOPN,
    METHOD_RANDOM,    
]
out_dir = eval_dir
in_dir, out_dir

('../QCB_chems/results', 'evaluation_QCB')

In [13]:
intensity_threshold = 0.5

#### Load pre-processed QCB chemicals

In [14]:
fullscan_file = '../fullscan_QCB.mzML'

In [15]:
# min_roi_intensity = 0
# min_roi_length = 0

# min_roi_intensity = 500
# min_roi_length = 0
# at_least_one_point_above = 5000

min_roi_intensity = 0
min_roi_length = 3
at_least_one_point_above = 1000

In [16]:
filename = '../datasets_%d_%d_%d.p' % (min_roi_intensity, min_roi_length, at_least_one_point_above)

if exists(filename):
    chemicals = load_obj(filename)
    print(len(chemicals))
else:
    rp = RoiBuilderParams(min_roi_intensity=min_roi_intensity, min_roi_length=min_roi_length, 
                   at_least_one_point_above=at_least_one_point_above)
    cm = ChemicalMixtureFromMZML(fullscan_file, roi_params=rp)
    chemicals = cm.sample(None, 2, source_polarity=ionisation_mode)
    print(len(chemicals))
    save_obj(chemicals, filename)

43107


#### Filter chemicals by mz and RT range

In [17]:
filtered = []
for chem in chemicals:
    if (min_mz < chem.isotopes[0][0] < max_mz) and (min_rt < chem.rt < max_rt):
        filtered.append(chem)
        
len(filtered)

43060

In [18]:
filtered_chem_list = [filtered]

#### Test different methods

In [19]:
copy_params = dict(params)
copy_params['env']['rt_tol'] = rt_tol

for method in methods:
    banner = 'method = %s max_peaks = %d rt_tol = %d' % (method, max_peaks, rt_tol)
    print(banner)
    print()
    
    copy_params = dict(params)
    if method == 'PPO':
        fname = os.path.join(in_dir, '%s_%s.zip' % (env_name, method))
        model = PPO.load(fname)
        copy_params['env']['rt_tol'] = rt_tol
    elif method == 'DQN':
        fname = os.path.join(in_dir, '%s_%s.zip' % (env_name, method))
        model = DQN.load(fname)
        copy_params['env']['rt_tol'] = rt_tol        
    else:
        model = None
        copy_params['env']['rt_tol'] = small_rt_tol        

    env = DDAEnv(max_peaks, copy_params)
    run_method(env_name, copy_params, max_peaks, filtered_chem_list, method, out_dir, mzml_prefix='QCB',
               N=10, min_ms1_intensity=min_ms1_intensity, model=model, print_reward=True, print_eval=True)
    print()

method = PPO max_peaks = 200 rt_tol = 120


Episode 0 (43060 chemicals)
steps	 500 	total rewards	 77.74605366747605
steps	 1000 	total rewards	 213.621310957099
steps	 1500 	total rewards	 383.3587777463258
steps	 2000 	total rewards	 542.9586966533096
steps	 2500 	total rewards	 730.8005825040915
steps	 3000 	total rewards	 928.5281793652521
steps	 3500 	total rewards	 1052.3996761449716
steps	 4000 	total rewards	 1115.9573814687687
steps	 4500 	total rewards	 1162.2524768063613
steps	 5000 	total rewards	 1204.1641258874433
steps	 5500 	total rewards	 1244.195879722733
Finished after 5768 timesteps with total reward 1269.052996929288
{'coverage_prop': '0.080', 'intensity_prop': '0.048', 'ms1/ms2 ratio': '0.330', 'efficiency': '0.790', 'TP': '1470', 'FP': '610', 'FN': '40980', 'precision': '0.707', 'recall': '0.035', 'f1': '0.066'}

method = topN max_peaks = 200 rt_tol = 120


Episode 0 (43060 chemicals)
steps	 500 	total rewards	 27.30326929199802
steps	 1000 	total rewards	 138.60

#### Test classic Top-N controller in ViMMS

In [20]:
mass_spec = IndependentMassSpectrometer(ionisation_mode, filtered)

In [23]:
mz_tol, rt_tol, min_ms1_intensity

(10, 120, 5000)

In [21]:
controller = TopNController(ionisation_mode, N, isolation_window, mz_tol, rt_tol, min_ms1_intensity)
env = Environment(mass_spec, controller, min_rt, max_rt, progress_bar=True, out_dir=out_dir, 
                  out_file='QCB_TopN_controller.mzML')
env.run()

  0%|          | 0/1440 [00:00<?, ?it/s]

#### Evaluation from mzML

Evaluation against the list of peaks picked from the fullscan QCB files.

There are two sets of parameters used for the peak picking: 'before' and 'after'.
'After' is more strict than 'before', with higher thresholds on intensity etc.

In [24]:
fullscan_path = os.path.abspath('../fullscan_QCB.mzML')
fullscan_path

'/Users/joewandy/Work/git/vimms-gym/notebooks/fullscan_QCB.mzML'

### 'Before' results

In [25]:
aligned_file = os.path.abspath('../fullscan_QCB_box_before.csv')
aligned_file

'/Users/joewandy/Work/git/vimms-gym/notebooks/fullscan_QCB_box_before.csv'

In [26]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_PPO_0.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.33080465998374425], [0.22302441365564743])

In [27]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_TopN_0.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.27363858033053373], [0.20956085121182907])

In [28]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_TopN_controller.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.4191276076943918], [0.2742782875513426])

In [29]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_random_0.mzML'),  
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.12408561365483609], [0.2754201422291434])

### 'After' results

In [30]:
aligned_file = os.path.abspath('../fullscan_QCB_box_after.csv')
aligned_file

'/Users/joewandy/Work/git/vimms-gym/notebooks/fullscan_QCB_box_after.csv'

In [31]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_PPO_0.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.7218045112781954], [0.37702584665655897])

In [32]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_TopN_0.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.5852130325814536], [0.4294251485704215])

In [33]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_TopN_controller.mzML'),
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.7969924812030075], [0.34534735857506416])

In [34]:
mzml_map = {fullscan_path: [
    os.path.join(out_dir, 'QCB_random_0.mzML'),  
]}
res = evaluate_real(aligned_file, mzml_map)
res['coverage_proportion'], res['intensity_proportion']

([0.49122807017543857], [0.5240316786871385])